In [ ]:
# lseg.data取得 -> weekly_intensity 出力まで（pipeline抜粋）

from pathlib import Path
import sys
import pandas as pd
import lseg.data as ld

PROJECT_ROOT = Path("/Users/kencharoff/workspace/projects/thematic-topic")
PROJECT_SRC = PROJECT_ROOT / "src"
if str(PROJECT_SRC) not in sys.path:
    sys.path.append(str(PROJECT_SRC))

from thematic_topic.config import load_config, PipelineConfig
from thematic_topic.headlines import fetch_headlines
from thematic_topic.preprocess import clean_headlines
from thematic_topic.embeddings import encode_texts_with_cache
from thematic_topic.dedup import deduplicate_headlines
from thematic_topic.topics import fit_topic_model, transform_topics
from thematic_topic.intensity import build_topic_intensity


def _select_fit_sample(df: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    out = df.copy()
    out["timestamp"] = pd.to_datetime(out["timestamp"], utc=True, errors="coerce")

    fit_start = cfg.topic_model.fit_start
    fit_end = cfg.topic_model.fit_end
    if fit_start:
        out = out[out["timestamp"] >= pd.to_datetime(fit_start, utc=True)]
    if fit_end:
        out = out[out["timestamp"] <= pd.to_datetime(fit_end, utc=True)]

    if out.empty:
        raise ValueError("BERTopic学習期間に該当するニュースがありません。")
    return out.reset_index(drop=True)


cfg = load_config(path=PROJECT_ROOT / "config/default.yaml", project_root=PROJECT_ROOT)

ld.open_session()
try:
    # 1) LSEG取得 + 正規化
    raw_df, normalized_df = fetch_headlines(cfg)

    # 2) 前処理
    clean_df = clean_headlines(
        normalized_df,
        mode=cfg.preprocess.low_information_mode,
        min_chars=cfg.preprocess.min_headline_chars,
    )

    # 3) embedding（本文主入力: model_input_text）
    cache_path = cfg.resolve_path(cfg.embedding.headline_cache_path)
    emb_meta, all_embeddings = encode_texts_with_cache(
        ids=clean_df["news_id"],
        texts=clean_df["model_input_text"],
        cache_path=cache_path,
        model_name=cfg.embedding.model_name,
        batch_size=cfg.embedding.batch_size,
        normalize_embeddings=cfg.embedding.normalize_embeddings,
    )

    # 4) 重複抑制
    dedup_df, dedup_log = deduplicate_headlines(
        clean_df,
        embeddings=all_embeddings,
        similarity_threshold=cfg.dedup.similarity_threshold,
        window_hours=cfg.dedup.window_hours,
    )

    # 5) dedup後embedding再マッピング
    clean_pos = clean_df.reset_index()[["index", "news_id"]]
    dedup_indexer = (
        dedup_df[["news_id"]]
        .merge(clean_pos, on="news_id", how="left")["index"]
        .to_numpy(dtype=int)
    )
    dedup_emb = all_embeddings[dedup_indexer]

    # 6) BERTopic fit/transform
    fit_df = _select_fit_sample(dedup_df, cfg)
    dedup_pos = dedup_df.reset_index()[["index", "news_id"]]
    fit_idx = (
        fit_df[["news_id"]]
        .merge(dedup_pos, on="news_id", how="left")["index"]
        .to_numpy(dtype=int)
    )
    fit_emb = dedup_emb[fit_idx]

    topic_model, fit_tables = fit_topic_model(
        fit_df, fit_emb, cfg.topic_model, text_col="model_input_text"
    )
    topic_assignments = transform_topics(
        dedup_df, topic_model, dedup_emb, text_col="model_input_text"
    )

    # 7) topic強度（daily/weekly）
    daily_intensity, weekly_intensity, outlier_stats = build_topic_intensity(
        topic_assignments,
        weekly_rule=cfg.topic_intensity.weekly_rule,       # "W-FRI"
        ewma_span=cfg.topic_intensity.ewma_span,           # 5
        lookback=cfg.topic_intensity.lookback,             # 20
        z_threshold=cfg.topic_intensity.z_threshold,       # 1.0
        aggregate_timezone=cfg.topic_intensity.aggregate_timezone,  # "Asia/Tokyo"
    )
finally:
    ld.close_session()

print("weekly_intensity shape:", weekly_intensity.shape)
display(weekly_intensity.head(20))
display(outlier_stats)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style="whitegrid")
np.random.seed(42)

# ----------------------------
# 1) ダミーデータ作成
# ----------------------------
topic_summary = pd.DataFrame({
    "topic_id": [0, 1, 2, 3],
    "topic_count": [28, 22, 18, 14],
    "top_words": [
        ["半導体", "AI", "データセンター"],
        ["金利", "銀行", "利ざや"],
        ["防衛", "政策", "予算"],
        ["再エネ", "電力", "蓄電池"],
    ],
    "representative_headlines": [
        ["GPU需要増で装置株高", "生成AI向け投資拡大"],
        ["長期金利上昇で銀行株堅調", "地銀再編観測"],
        ["防衛費増額を巡る議論", "装備調達前倒し"],
        ["再エネ導入加速へ制度改定", "電力需給逼迫懸念"],
    ],
})

n_days = 70
dates = pd.date_range("2024-01-01", periods=n_days, freq="D")

rows = []
for tid in topic_summary["topic_id"]:
    base = 0.25 + 0.04 * tid
    trend = np.linspace(0.0, 0.15 + 0.03 * tid, n_days)
    season = 0.06 * np.sin(np.linspace(0, 4 * np.pi, n_days) + tid)
    noise = np.random.normal(0, 0.015, n_days)
    prob_sum = np.clip(base + trend + season + noise, 0.01, None)

    ewma = pd.Series(prob_sum).ewm(span=5, adjust=False).mean().to_numpy()
    ma = pd.Series(prob_sum).rolling(20, min_periods=5).mean().shift(1)
    sd = pd.Series(prob_sum).rolling(20, min_periods=5).std(ddof=0).shift(1).replace(0, np.nan)
    z = (pd.Series(prob_sum) - ma) / sd

    run = 0
    persistence = []
    for v in np.nan_to_num(z.to_numpy(), nan=-999):
        run = run + 1 if v > 1.0 else 0
        persistence.append(run)

    for d, ps, e, zz, p in zip(dates, prob_sum, ewma, z, persistence):
        rows.append(
            {
                "date": d,
                "topic_id": int(tid),
                "topic_prob_sum": float(ps),
                "topic_ewma": float(e),
                "topic_z": float(zz) if not np.isnan(zz) else np.nan,
                "topic_persistence": int(p),
            }
        )

topic_daily = pd.DataFrame(rows)

# ----------------------------
# 2) weekly_intensity 作成 (W-FRI)
# ----------------------------
topic_weekly = (
    topic_daily.set_index("date")
    .groupby("topic_id")[["topic_prob_sum"]]
    .resample("W-FRI")
    .sum()
    .reset_index()
)

topic_weekly["topic_ewma"] = topic_weekly.groupby("topic_id")["topic_prob_sum"].transform(
    lambda s: s.ewm(span=5, adjust=False).mean()
)
topic_weekly["topic_z"] = topic_weekly.groupby("topic_id")["topic_prob_sum"].transform(
    lambda s: (s - s.rolling(20, min_periods=5).mean().shift(1))
    / s.rolling(20, min_periods=5).std(ddof=0).shift(1).replace(0, np.nan)
)

# persistence
pvals = []
for _, g in topic_weekly.groupby("topic_id"):
    run = 0
    vals = []
    for v in np.nan_to_num(g["topic_z"].to_numpy(), nan=-999):
        run = run + 1 if v > 1.0 else 0
        vals.append(run)
    pvals.extend(vals)
topic_weekly["topic_persistence"] = pvals

# ----------------------------
# 3) 各トピック内訳
# ----------------------------
topic_breakdown = (
    topic_daily.groupby("topic_id", as_index=False)
    .agg(
        count=("topic_id", "size"),
        prob_sum=("topic_prob_sum", "sum"),
        ewma_latest=("topic_ewma", "last"),
        z_latest=("topic_z", "last"),
        persistence_latest=("topic_persistence", "last"),
    )
    .sort_values("prob_sum", ascending=False)
)

print("=== topic_weekly head ===")
display(topic_weekly.head(12))
print("=== topic_breakdown ===")
display(topic_breakdown)

# ----------------------------
# 4) weekly intensity 時系列プロット
# ----------------------------
plt.figure(figsize=(11, 4.2))
for tid, g in topic_weekly.groupby("topic_id"):
    plt.plot(g["date"], g["topic_prob_sum"], marker="o", linewidth=1.8, label=f"topic {tid}")
plt.title("Weekly Topic Intensity (topic_prob_sum, W-FRI)")
plt.xlabel("date")
plt.ylabel("prob_sum")
plt.legend(loc="upper left", ncols=2)
plt.tight_layout()
plt.show()

# ----------------------------
# 5) topic内訳 棒グラフ
# ----------------------------
plt.figure(figsize=(8.5, 4.2))
plt.bar(topic_breakdown["topic_id"].astype(str), topic_breakdown["prob_sum"])
plt.title("Topic Breakdown (total prob_sum)")
plt.xlabel("topic_id")
plt.ylabel("total prob_sum")
plt.tight_layout()
plt.show()

# ----------------------------
# 6) クラスタ可視化（UMAP 2D + 類似度ヒートマップ）
# ----------------------------
texts = [
    " | ".join(words + heads)
    for words, heads in zip(topic_summary["top_words"], topic_summary["representative_headlines"])
]

# 埋め込み: まずSentenceTransformer(ローカル優先)、失敗時TF-IDF fallback
embed_source = "sentence-transformer"
try:
    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(
        "paraphrase-multilingual-MiniLM-L12-v2",
        device="cpu",
        cache_folder=str(Path("/private/tmp/hf_cache")),
    )
    emb = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
except Exception:
    from sklearn.feature_extraction.text import TfidfVectorizer

    embed_source = "tfidf-fallback"
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
    emb = vec.fit_transform(texts).toarray()

# 2D圧縮: UMAP優先、なければPCA fallback
reducer_name = "UMAP"
try:
    from umap import UMAP

    reducer = UMAP(n_neighbors=3, n_components=2, min_dist=0.0, metric="cosine", random_state=42)
    coords = reducer.fit_transform(emb)
except Exception:
    from sklearn.decomposition import PCA

    reducer_name = "PCA-fallback"
    coords = PCA(n_components=2, random_state=42).fit_transform(emb)

cluster_df = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1], "topic_id": topic_summary["topic_id"].tolist()})

plt.figure(figsize=(7.2, 6.0))
palette = sns.color_palette("tab10", len(cluster_df))
for i, row in cluster_df.iterrows():
    plt.scatter(row["x"], row["y"], s=180, color=palette[i], edgecolor="black")
    plt.text(row["x"] + 0.02, row["y"] + 0.02, f"topic {int(row['topic_id'])}", fontsize=10)
plt.title(f"Topic Cluster Plot ({reducer_name}, emb={embed_source})")
plt.xlabel("dim-1")
plt.ylabel("dim-2")
plt.tight_layout()
plt.show()

sim = cosine_similarity(emb)
sim_df = pd.DataFrame(
    sim,
    index=[f"topic {t}" for t in topic_summary["topic_id"]],
    columns=[f"topic {t}" for t in topic_summary["topic_id"]],
)

plt.figure(figsize=(6.6, 5.6))
sns.heatmap(sim_df, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1)
plt.title(f"Topic Similarity Heatmap (emb={embed_source})")
plt.tight_layout()
plt.show()
